# Fase 1 — Evaluación de geocodificación inversa

**Proyecto:** Panel de viaje para coche (`tragabytes/panel-viaje`)
**Fase del plan:** 1.1 — Geocodificación inversa (coordenadas → municipio / provincia / CCAA)

## Qué hace este notebook

Prueba **dos APIs gratuitas de geocodificación inversa** —Nominatim y Photon— con **12 coordenadas repartidas por España**: una por cada comunidad autónoma de las 10 que elegí como muestra representativa, más dos casos especiales que marca el plan:

1. Un punto sobre la autovía **A-2 cerca de Medinaceli**, para ver si alguna API devuelve el nombre de la carretera.
2. Un punto en un **límite provincial** (costa entre Valencia y Castellón), para ver si hay ambigüedad.

Para cada coordenada se mide:

- Qué municipio, provincia y comunidad autónoma devuelve cada API.
- Si hay nombre de carretera.
- Latencia de la petición.
- Tamaño de la respuesta en bytes.

Al final se genera una **tabla comparativa** y un **resumen con las medias**, que es lo que hay que pegar en la ficha del documento de seguimiento.

## Cómo ejecutarlo

`Entorno de ejecución → Ejecutar todas`. Tarda unos 20 segundos (Nominatim exige 1 petición/segundo, de ahí el cuello de botella).

## Importante

- **Nominatim** requiere un `User-Agent` identificativo por su política de uso. Ya está puesto.
- **La latencia medida aquí no es la del móvil en el coche.** Colab corre en servidores de Google con fibra; el móvil irá por datos compartidos. La latencia real se medirá en la fase 5. Lo que sí es fiable en Colab: qué campos devuelve cada API, calidad de los datos y comparativa entre ambas.


In [ ]:
import requests
import time
import pandas as pd
from IPython.display import display

# 10 coordenadas (una por CCAA) + 2 casos especiales del plan
COORDS = [
    # (nombre_esperado, ccaa_esperada, lat, lon, nota)
    ("Ronda",                    "Andalucía",          36.7428, -5.1658, "pueblo mediano"),
    ("Albarracín",               "Aragón",             40.4064, -1.4433, "pueblo pequeño histórico"),
    ("Cangas de Onís",           "Asturias",           43.3509, -5.1299, "pueblo pequeño"),
    ("Valldemossa",              "Islas Baleares",     39.7104,  2.6231, "pueblo pequeño"),
    ("Teguise",                  "Canarias",           29.0607,-13.5626, "pueblo pequeño"),
    ("Santillana del Mar",       "Cantabria",          43.3929, -4.1085, "pueblo pequeño turístico"),
    ("Almagro",                  "Castilla-La Mancha", 38.8898, -3.7125, "pueblo pequeño"),
    ("La Alberca",               "Castilla y León",    40.4919, -6.1122, "pueblo pequeño"),
    ("Besalú",                   "Cataluña",           42.1993,  2.6985, "pueblo pequeño"),
    ("Chinchón",                 "Madrid",             40.1394, -3.4266, "pueblo pequeño"),
    # Casos especiales del plan:
    ("A-2 cerca de Medinaceli",  "Castilla y León",    41.1710, -2.4300, "punto sobre autovía"),
    ("Límite Valencia/Castellón","C. Valenciana",      39.7500, -0.2300, "zona límite provincial"),
]

print(f"Total de coordenadas a probar: {len(COORDS)}")


## 1. Prueba de Nominatim

Endpoint: `https://nominatim.openstreetmap.org/reverse`

Política: máximo 1 petición/segundo, `User-Agent` obligatorio.

In [ ]:
NOMINATIM_URL = "https://nominatim.openstreetmap.org/reverse"
HEADERS = {
    "User-Agent": "panel-viaje-test/0.1 (github.com/tragabytes/panel-viaje)"
}

nominatim_rows = []
for name, ccaa, lat, lon, nota in COORDS:
    params = {
        "lat": lat, "lon": lon,
        "format": "json",
        "accept-language": "es",
        "zoom": 14,
        "addressdetails": 1,
    }
    try:
        t0 = time.time()
        r = requests.get(NOMINATIM_URL, params=params, headers=HEADERS, timeout=15)
        ms = (time.time() - t0) * 1000
        data = r.json()
        addr = data.get("address", {})

        muni = (addr.get("city") or addr.get("town") or addr.get("village")
                or addr.get("municipality") or addr.get("hamlet") or "—")
        prov = addr.get("province") or addr.get("county") or "—"
        state = addr.get("state") or "—"
        road = addr.get("road") or "—"

        nominatim_rows.append({
            "punto": name,
            "ccaa_esperada": ccaa,
            "muni": muni,
            "prov": prov,
            "state (CCAA)": state,
            "road": road,
            "ms": round(ms),
            "bytes": len(r.content),
        })
        print(f"✓ {name:28s} → {muni} / {prov} / {state}  [{round(ms)} ms]")
    except Exception as e:
        nominatim_rows.append({"punto": name, "ccaa_esperada": ccaa, "error": str(e)})
        print(f"✗ {name}: {e}")
    time.sleep(1.1)  # respetar política de 1 req/s

df_nom = pd.DataFrame(nominatim_rows)
print("\n── Tabla Nominatim ──")
display(df_nom)


## 2. Prueba de Photon

Endpoint: `https://photon.komoot.io/reverse`

No hay límite documentado estricto, pero se usa una pausa suave entre peticiones por cortesía.

In [ ]:
PHOTON_URL = "https://photon.komoot.io/reverse"

photon_rows = []
for name, ccaa, lat, lon, nota in COORDS:
    params = {"lat": lat, "lon": lon, "lang": "es"}
    try:
        t0 = time.time()
        r = requests.get(PHOTON_URL, params=params, timeout=15)
        ms = (time.time() - t0) * 1000
        data = r.json()
        feats = data.get("features", [])

        if feats:
            p = feats[0].get("properties", {})
            muni = (p.get("city") or p.get("town") or p.get("village")
                    or p.get("locality") or p.get("name") or "—")
            prov = p.get("county") or "—"
            state = p.get("state") or "—"
            road = p.get("street") or (p.get("name") if p.get("osm_key") == "highway" else "—")

            photon_rows.append({
                "punto": name,
                "ccaa_esperada": ccaa,
                "muni": muni,
                "prov": prov,
                "state (CCAA)": state,
                "road": road,
                "osm_key": p.get("osm_key"),
                "ms": round(ms),
                "bytes": len(r.content),
            })
            print(f"✓ {name:28s} → {muni} / {prov} / {state}  [{round(ms)} ms]")
        else:
            photon_rows.append({"punto": name, "ccaa_esperada": ccaa, "error": "sin resultados"})
            print(f"✗ {name}: sin resultados")
    except Exception as e:
        photon_rows.append({"punto": name, "ccaa_esperada": ccaa, "error": str(e)})
        print(f"✗ {name}: {e}")
    time.sleep(0.3)

df_pho = pd.DataFrame(photon_rows)
print("\n── Tabla Photon ──")
display(df_pho)


## 3. Comparativa lado a lado

Mismo punto, las dos APIs, para ver de un vistazo qué devuelve cada una.

In [ ]:
# Construir tabla comparativa
comp_rows = []
for (name, ccaa, lat, lon, nota), n, p in zip(COORDS, nominatim_rows, photon_rows):
    comp_rows.append({
        "punto": name,
        "esperado (CCAA)": ccaa,
        "NOM muni":  n.get("muni", "ERROR"),
        "NOM CCAA":  n.get("state (CCAA)", "ERROR"),
        "NOM road":  n.get("road", "ERROR"),
        "NOM ms":    n.get("ms", "—"),
        "PHO muni":  p.get("muni", "ERROR"),
        "PHO CCAA":  p.get("state (CCAA)", "ERROR"),
        "PHO road":  p.get("road", "ERROR"),
        "PHO ms":    p.get("ms", "—"),
    })

df_comp = pd.DataFrame(comp_rows)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
display(df_comp)


## 4. Resumen

Medias de latencia y tamaño, aciertos, y observaciones para pegar en la ficha del documento de seguimiento.

In [ ]:
def resumen(rows, label):
    ok = [r for r in rows if "error" not in r]
    errs = [r for r in rows if "error" in r]
    if not ok:
        print(f"{label}: 0/{len(rows)} OK — todas las peticiones fallaron")
        return
    avg_ms  = sum(r["ms"]    for r in ok) / len(ok)
    avg_sz  = sum(r["bytes"] for r in ok) / len(ok)
    max_ms  = max(r["ms"]    for r in ok)
    min_ms  = min(r["ms"]    for r in ok)
    # Aciertos de CCAA: comparación tolerante (contiene)
    ccaa_ok = sum(
        1 for r in ok
        if r["ccaa_esperada"].split(" ")[0].lower() in str(r.get("state (CCAA)", "")).lower()
        or str(r.get("state (CCAA)", "")).lower() in r["ccaa_esperada"].lower()
    )
    road_detectado = sum(1 for r in ok if r.get("road", "—") not in ("—", None, ""))

    print(f"── {label} ──")
    print(f"  Peticiones OK:       {len(ok)}/{len(rows)}")
    print(f"  CCAA correcta:       {ccaa_ok}/{len(ok)}")
    print(f"  Road detectado:      {road_detectado}/{len(ok)}")
    print(f"  Latencia media:      {avg_ms:.0f} ms  (min {min_ms}, max {max_ms})")
    print(f"  Tamaño medio:        {avg_sz:.0f} bytes")
    if errs:
        print(f"  Errores:             {len(errs)}")
        for e in errs:
            print(f"    - {e['punto']}: {e.get('error')}")
    print()

resumen(nominatim_rows, "NOMINATIM")
resumen(photon_rows,    "PHOTON")

print("── Veredicto rápido ──")
print("Revisa las tablas de arriba: los campos más importantes son `muni`, `state (CCAA)` y `road`.")
print("Si una API falla en devolver la CCAA o el municipio de un pueblo pequeño, anótalo como limitación.")
print("La fila de A-2 cerca de Medinaceli es la que dice si sirven para detectar carretera.")
